In [1]:
import torch
import torch.nn.functional as F
from torch_geometric.nn import GCNConv
from sklearn.metrics import accuracy_score

d:\GitHub\FedGNN-CKD\fedgnn_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Check Device (CPU/GPU)

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cpu')

Define the GCN Model (CORE MODEL)

In [3]:
class GCN(torch.nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super(GCN, self).__init__()
        self.conv1 = GCNConv(input_dim, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, output_dim)

    def forward(self, data):
        x, edge_index = data.x, data.edge_index

        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = F.dropout(x, training=self.training)

        x = self.conv2(x, edge_index)
        return x

Training Function (Reusable for Any Hospital)

In [4]:
def train_local_model(graph, epochs=100, lr=0.01):
    graph = graph.to(device)

    model = GCN(
        input_dim=graph.num_node_features,
        hidden_dim=32,
        output_dim=2
    ).to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = torch.nn.CrossEntropyLoss()

    model.train()
    for epoch in range(epochs):
        optimizer.zero_grad()

        out = model(graph)
        loss = criterion(out, graph.y)

        loss.backward()
        optimizer.step()

        if epoch % 20 == 0:
            print(f"Epoch {epoch:03d} | Loss: {loss:.4f}")

    return model


Train Models at Each Hospital

In [5]:
model_A = train_local_model(graph_A)
model_B = train_local_model(graph_B)
model_C = train_local_model(graph_C)

NameError: name 'graph_A' is not defined